# Week 02 · Day 04 — LLMOps

**Topics:** Structured logging · LangSmith tracing · Cost tracking with tiktoken · Latency benchmarking


In [ ]:
%pip install -q openai tiktoken langsmith

In [ ]:
import os, time, uuid, json, logging
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Structured logging setup
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(message)s',
)
logger = logging.getLogger(__name__)

## 1 · Structured Logging for LLM Calls

Every LLM call should emit a structured log record with enough information to debug issues and compute cost without having to re-run.

In [ ]:
PRICING = {
    "gpt-4o":       {"input": 2.50, "output": 10.00},
    "gpt-4o-mini":  {"input": 0.15, "output": 0.60},
}

def llm_call_with_logging(
    messages: list[dict],
    model: str = "gpt-4o-mini",
    user_id: str = None,
    **kwargs,
) -> str:
    request_id = str(uuid.uuid4())[:8]
    t0 = time.perf_counter()

    try:
        response = client.chat.completions.create(
            model=model, messages=messages, **kwargs
        )
        latency_ms = (time.perf_counter() - t0) * 1000

        usage = response.usage
        p = PRICING.get(model, {"input": 0, "output": 0})
        cost_usd = (
            usage.prompt_tokens * p["input"] +
            usage.completion_tokens * p["output"]
        ) / 1_000_000

        log_record = {
            "request_id": request_id,
            "model": model,
            "user_id": user_id,
            "latency_ms": round(latency_ms, 1),
            "input_tokens": usage.prompt_tokens,
            "output_tokens": usage.completion_tokens,
            "cost_usd": round(cost_usd, 6),
            "finish_reason": response.choices[0].finish_reason,
        }
        logger.info(json.dumps(log_record))
        return response.choices[0].message.content

    except Exception as e:
        logger.error(json.dumps({
            "request_id": request_id,
            "error": str(e),
            "latency_ms": round((time.perf_counter() - t0) * 1000, 1),
        }))
        raise

result = llm_call_with_logging(
    messages=[{"role": "user", "content": "What is 2 + 2?"}],
    user_id="user-abc",
    temperature=0,
    max_tokens=20,
)
print(f"Answer: {result}")

## 2 · LangSmith Tracing

LangSmith traces every LLM call, tool call, and chain invocation. Add `@traceable` to any function to include it in the trace tree.

In [ ]:
# Set up LangSmith (requires LANGCHAIN_API_KEY)
langsmith_enabled = bool(os.getenv("LANGCHAIN_API_KEY"))

if langsmith_enabled:
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_PROJECT"] = "llm-course-demo"
    print("LangSmith tracing enabled")
else:
    print("Set LANGCHAIN_API_KEY to enable LangSmith tracing")
    print("Sign up free at: https://smith.langchain.com")

In [ ]:
from langsmith import traceable
import chromadb

# Mock ChromaDB for demo
chroma = chromadb.Client()
col = chroma.get_or_create_collection("llmops_demo")

def embed(texts):
    r = client.embeddings.create(input=texts, model="text-embedding-3-small")
    return [item.embedding for item in r.data]

docs = ["Python uses indentation for blocks.", "FastAPI is built on Starlette.", "LangSmith traces LLM calls."]
col.add(documents=docs, embeddings=embed(docs), ids=[f"d{i}" for i in range(len(docs))])

@traceable(name="retrieve_context")
def retrieve(query: str, n: int = 3) -> list[str]:
    q_emb = embed([query])[0]
    results = col.query(query_embeddings=[q_emb], n_results=n)
    return results["documents"][0]

@traceable(name="generate_answer")
def generate(question: str, context: list[str]) -> str:
    ctx = "\n".join(context)
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Answer using only the context."},
            {"role": "user", "content": f"Context: {ctx}\n\nQuestion: {question}"},
        ],
        temperature=0,
        max_tokens=150,
    )
    return r.choices[0].message.content

@traceable(name="rag_pipeline", tags=["rag", "demo"])
def rag_pipeline(question: str) -> str:
    context = retrieve(question)
    return generate(question, context)

answer = rag_pipeline("How does Python define code blocks?")
print(f"Answer: {answer}")

## 3 · Cost Tracking with tiktoken

In [ ]:
import tiktoken

def estimate_cost(messages: list[dict], model: str = "gpt-4o-mini", expected_output_tokens: int = 200) -> dict:
    enc = tiktoken.encoding_for_model(model)
    input_tokens = sum(len(enc.encode(m["content"])) + 4 for m in messages) + 2
    p = PRICING.get(model, {"input": 0, "output": 0})
    input_cost = input_tokens * p["input"] / 1_000_000
    output_cost = expected_output_tokens * p["output"] / 1_000_000
    return {
        "model": model,
        "input_tokens": input_tokens,
        "expected_output_tokens": expected_output_tokens,
        "estimated_cost_usd": round(input_cost + output_cost, 6),
    }

# Compare cost across models
messages = [
    {"role": "system", "content": "You are a helpful assistant that answers questions about Python."},
    {"role": "user", "content": "Explain the difference between lists and tuples in Python with examples."},
]

print(f"{'Model':<15} {'Input tok':>10} {'Estimated cost':>15}")
print("-" * 45)
for model in PRICING:
    estimate = estimate_cost(messages, model=model)
    print(f"{model:<15} {estimate['input_tokens']:>10} ${estimate['estimated_cost_usd']:>14.6f}")

In [ ]:
# Track cumulative cost across a session
class CostTracker:
    def __init__(self):
        self.calls = []

    def record(self, model: str, input_tokens: int, output_tokens: int):
        p = PRICING.get(model, {"input": 0, "output": 0})
        cost = (input_tokens * p["input"] + output_tokens * p["output"]) / 1_000_000
        self.calls.append({"model": model, "input": input_tokens, "output": output_tokens, "cost": cost})

    def summary(self):
        total_cost = sum(c["cost"] for c in self.calls)
        total_input = sum(c["input"] for c in self.calls)
        total_output = sum(c["output"] for c in self.calls)
        return {
            "calls": len(self.calls),
            "total_input_tokens": total_input,
            "total_output_tokens": total_output,
            "total_cost_usd": round(total_cost, 4),
        }

tracker = CostTracker()

# Simulate a session with multiple calls
for i in range(5):
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": f"Name one Python library for data science. Be brief. ({i})"}],
        max_tokens=30,
        temperature=0,
    )
    tracker.record("gpt-4o-mini", r.usage.prompt_tokens, r.usage.completion_tokens)

print("Session summary:")
print(json.dumps(tracker.summary(), indent=2))

## 4 · Latency Benchmarking

In [ ]:
import statistics

def benchmark_latency(model: str, prompt: str, n_runs: int = 5, max_tokens: int = 100) -> dict:
    latencies = []
    ttft_values = []  # time to first token

    for _ in range(n_runs):
        t0 = time.perf_counter()
        first_token = None

        stream = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            stream=True,
            max_tokens=max_tokens,
            temperature=0,
        )
        for chunk in stream:
            if first_token is None and chunk.choices[0].delta.content:
                first_token = time.perf_counter() - t0

        total = time.perf_counter() - t0
        latencies.append(total)
        if first_token:
            ttft_values.append(first_token)

    return {
        "model": model,
        "n_runs": n_runs,
        "p50_ms": round(statistics.median(latencies) * 1000, 0),
        "p95_ms": round(sorted(latencies)[int(n_runs * 0.95)] * 1000, 0),
        "mean_ms": round(statistics.mean(latencies) * 1000, 0),
        "ttft_p50_ms": round(statistics.median(ttft_values) * 1000, 0) if ttft_values else None,
    }

prompt = "List 3 Python best practices."
result = benchmark_latency("gpt-4o-mini", prompt, n_runs=3)
print(json.dumps(result, indent=2))

## 5 · Key Metrics to Monitor in Production

| Metric | Alert threshold | What it signals |
|--------|-----------------|-----------------|
| p95 latency | > 5s | Model slow, network issue |
| Error rate | > 1% | Rate limits or API outage |
| Cost per request | > 2× baseline | Prompt injection or runaway loop |
| Output token ratio | > 0.8 max_tokens | Answers being truncated |
| Cache hit rate | < 20% (if caching) | Cache keys not matching |

In [ ]:
# Latency optimization checklist
optimizations = [
    ("Use streaming", "Reduces perceived latency from 3s to instant first token"),
    ("Use gpt-4o-mini", "3-4x faster than gpt-4o, 10-20x cheaper — use for 80% of requests"),
    ("Semantic cache", "Serve repeated queries from cache — 0ms latency"),
    ("Async client", "Run concurrent requests without blocking"),
    ("Reduce prompt size", "Every 100 tokens saved = ~50ms faster"),
    ("Prompt caching (Anthropic)", "50% discount on cached input tokens for long system prompts"),
]

print("Latency optimization checklist:")
for opt, reason in optimizations:
    print(f"  ☐ {opt:<30} — {reason}")

## Summary

- Log every LLM call: request_id, latency_ms, tokens, cost, finish_reason.
- `@traceable` wraps any function in a LangSmith trace — nested calls form a tree.
- Estimate cost before batch jobs with tiktoken; track cumulative cost with a `CostTracker`.
- Benchmark with `p50` + `p95` latency and TTFT (time-to-first-token) separately.
- Cost anomaly alerts catch prompt injection and runaway loops early.

**Next:** [Deployment](week02-day04-deployment.ipynb)